# Query 示例

使用 `FactorQuery` 定义 DolphinDB 因子查询，并通过 `execute_query`
按需下载结果。

## Goal

1. 查询两只股票的 `close` 和 `vol`。
2. 在 DolphinDB 中计算单日收益率。
3. 仅保留收益率大于 0 的记录。
4. 使用 `with` 管理结果持有的 DolphinDB session。

## Setup

在项目根目录运行 `uv run jupyter lab`。DolphinDB 连接参数从项目的
`.env` 或 `DOLPHIN_HOST`、`DOLPHIN_PORT`、`DOLPHIN_USERNAME`、
`DOLPHIN_PASSWORD` 环境变量读取。

DolphinDB 中需要已经存在 CoreData 统一因子表及示例日期范围的数据。

In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import display

project_root = Path.cwd()
if project_root.name == "examples":
    project_root = project_root.parent
load_dotenv(project_root / ".env")
load_dotenv(project_root.parent / ".env")

from core import FactorQuery, execute_query
from core.utils import logger
logger.remove()
logger.add(sys.stderr, level="INFO")

2

## Steps

### 1. 定义查询

In [2]:
query_request = FactorQuery.model_validate(
    {
        "start_date": "2025-01-01",
        "end_date": "2025-03-31",
        "lookback": "10D",
        "codes": ["000001.SZ", "600000.SH"],
        "factors": ["close", "vol"],
        "derivatives": {
            "return_1d": {
                "type": "TS",
                "op": "unary.pct_change",
                "fields": {"col": "close"},
                "params": {"periods": 1},
            },
            "positive_return": {
                "type": "DIRECT",
                "op": "binary.gt",
                "fields": {"left": "return_1d", "right": 0},
                "params": {},
            },
        },
        "filters": ["positive_return"],
    }
)

print("codes:", query_request.codes)
print("factors:", query_request.factors)
print("derivatives:", list(query_request.derivatives))
print("filters:", query_request.filters)

codes: ['000001.SZ', '600000.SH']
factors: ['close', 'vol']
derivatives: ['return_1d', 'positive_return']
filters: ['positive_return']


### 2. 执行并按需下载

`execute_query` 返回 `QueryResult`。创建结果时数据仍在 session 中；
访问 `data` 或调用 `download()` 时才下载。退出 `with` 后 session 自动关闭。

In [3]:
with execute_query(query_request) as query_result:
    query_data = query_result.data
    print("session type:", type(query_result.session).__name__)
    display(query_data.head(10))
    display(query_result.source_data.head(10))

print("session closed:", query_result.closed)

2026-07-31 23:55:07.473 | INFO     | core.database.session:create_session:43 - DolphinDB: 1.13.198.44:8848
2026-07-31 23:55:07.577 | INFO     | core.apps.query.api:build_query_table:64 - session.run: 加载 query 模块
2026-07-31 23:55:07.606 | INFO     | core.database.session:has_session_variable:37 - session.run: 检查变量 coreQuerySourceData 是否存在
2026-07-31 23:55:07.612 | INFO     | core.apps.query.api:build_query_table:68 - session.run: 查询基础因子表 coreQuerySourceData
2026-07-31 23:55:07.622 | INFO     | core.apps.query.api:build_query_table:118 - session.run: 整理基础因子表 coreQuerySourceData
2026-07-31 23:55:07.631 | INFO     | core.apps.query.api:build_query_table:126 - session.run: 计算 coreQueryComputedData 并生成 coreQueryFilteredData
2026-07-31 23:55:07.637 | INFO     | core.apps.query.api:execute_query:154 - session.run: 生成查询最终结果 coreQueryData
2026-07-31 23:55:07.642 | SUCCESS  | core.apps.query.api:execute_query:164 - 因子查询已在 DolphinDB 会话中生成


session type: Session


,time,code,close,vol,return_1d,positive_return
0,2025-01-06,000001.SZ,11.44,1085536.30,0.005272,True
1,2025-01-07,000001.SZ,11.51,747862.88,0.006119,True
2,2025-01-14,000001.SZ,11.38,824628.95,0.016071,True
3,2025-01-15,000001.SZ,11.48,1031630.82,0.008787,True
4,2025-01-16,000001.SZ,11.57,872963.99,0.007840,True
5,2025-01-23,000001.SZ,11.32,1514920.27,0.020739,True
6,2025-01-24,000001.SZ,11.34,944944.21,0.001767,True
7,2025-01-27,000001.SZ,11.47,1151934.71,0.011464,True
8,2025-02-07,000001.SZ,11.38,1407492.62,0.001761,True
9,2025-02-10,000001.SZ,11.43,1026590.93,0.004394,True


,time,code,close,vol
0,2024-12-23,000001.SZ,11.73,1659404.76
1,2024-12-24,000001.SZ,11.86,1350836.91
2,2024-12-25,000001.SZ,11.92,1475282.94
3,2024-12-26,000001.SZ,11.86,1000074.70
4,2024-12-27,000001.SZ,11.83,1290012.28
5,2024-12-30,000001.SZ,11.95,1351846.36
6,2024-12-31,000001.SZ,11.70,1475367.33
7,2025-01-02,000001.SZ,11.43,1819596.99
8,2025-01-03,000001.SZ,11.38,1154680.44
9,2025-01-06,000001.SZ,11.44,1085536.30


session closed: True


## Checks

In [4]:
assert query_request.filters == ["positive_return"]
assert set(query_request.derivatives) == {"return_1d", "positive_return"}
assert query_result.closed
assert {"time", "code", "return_1d"}.issubset(query_data.columns)

print("Query example checks passed.")

Query example checks passed.


## Next Steps

- 调整 `codes` 和日期范围。
- 在 `derivatives` 中组合更多 DIRECT、TS 和 CS 算符。
- 使用 `query_result.download("任意 DOS 代码")` 读取同一 session 的其他结果。